# Mixed Provider Optimization Level Comparison

Compare transpilation optimization levels (0-3) across IBM and IQM quantum simulators.

## Backend Selection

In [ ]:
from iqm.qiskit_iqm import IQMFakeAdonis
from qiskit_ibm_runtime.fake_provider import FakeManilaV2
from qiskit import QuantumCircuit, transpile
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
import matplotlib.pyplot as plt
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Select one backend from each provider
iqm_backend = IQMFakeAdonis()
ibm_backend = FakeManilaV2()

print("Selected Backends:")
print("=" * 60)
print(f"IQM: {iqm_backend.name} ({iqm_backend.num_qubits} qubits)")
print(f"IBM: {ibm_backend.name} ({ibm_backend.num_qubits} qubits)")
print("=" * 60)

## Create Test Circuit

Create a 5-qubit circuit with entanglement and rotations.

In [ ]:
NUM_QUBITS = 5

qc = QuantumCircuit(NUM_QUBITS, NUM_QUBITS)

# Initial superposition
qc.h(range(NUM_QUBITS))

# Entanglement layer
qc.cz(0, 1)
qc.cz(1, 2)
qc.cz(2, 3)
qc.cz(3, 4)
qc.cz(0, 4)

# Rotations
for i in range(NUM_QUBITS):
    qc.rz(0.5, i)
    qc.ry(0.3, i)

# Additional entanglement
for i in range(NUM_QUBITS - 1):
    qc.cx(i, i + 1)

# Final layer
qc.h(range(NUM_QUBITS))
qc.measure(range(NUM_QUBITS), range(NUM_QUBITS))

print(f"Original Circuit: Depth={qc.depth()}, Gates={qc.size()}")

## Transpile with Different Optimization Levels

In [ ]:
OPTIMIZATION_LEVELS = [0, 1, 2, 3]
SHOTS = 4000

results = {'IQM': {}, 'IBM': {}}

# Process IQM backend
print("\nTranspiling for IQM...")
print("=" * 60)
for level in OPTIMIZATION_LEVELS:
    transpiled = transpile(qc, backend=iqm_backend, optimization_level=level, seed_transpiler=42)
    job = iqm_backend.run(transpiled, shots=SHOTS)
    results['IQM'][level] = {
        'circuit': transpiled,
        'counts': job.result().get_counts(),
        'depth': transpiled.depth(),
        'gates': transpiled.size()
    }
    print(f"Level {level}: Depth={transpiled.depth()}, Gates={transpiled.size()}")

# Process IBM backend
print("\nTranspiling for IBM...")
print("=" * 60)
for level in OPTIMIZATION_LEVELS:
    pm = generate_preset_pass_manager(optimization_level=level, backend=ibm_backend, seed_transpiler=42)
    transpiled = pm.run(qc)
    job = ibm_backend.run(transpiled, shots=SHOTS)
    results['IBM'][level] = {
        'circuit': transpiled,
        'counts': job.result().get_counts(),
        'depth': transpiled.depth(),
        'gates': transpiled.size()
    }
    print(f"Level {level}: Depth={transpiled.depth()}, Gates={transpiled.size()}")

print("\n" + "=" * 60)
print("Transpilation Complete!")
print("=" * 60)

## Compare Metrics

In [ ]:
# Create comparison visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('IBM vs IQM Optimization Comparison', fontsize=16, fontweight='bold')

colors_iqm = '#4ECDC4'
colors_ibm = '#FF6B6B'
levels_labels = [f'Level {l}' for l in OPTIMIZATION_LEVELS]

# Circuit Depth Comparison
ax1 = axes[0, 0]
iqm_depths = [results['IQM'][l]['depth'] for l in OPTIMIZATION_LEVELS]
ibm_depths = [results['IBM'][l]['depth'] for l in OPTIMIZATION_LEVELS]

x = np.arange(len(OPTIMIZATION_LEVELS))
width = 0.35

ax1.bar(x - width/2, iqm_depths, width, label='IQM', color=colors_iqm, alpha=0.8, edgecolor='black')
ax1.bar(x + width/2, ibm_depths, width, label='IBM', color=colors_ibm, alpha=0.8, edgecolor='black')
ax1.set_ylabel('Circuit Depth', fontsize=12, fontweight='bold')
ax1.set_xlabel('Optimization Level', fontsize=12, fontweight='bold')
ax1.set_title('Circuit Depth Comparison', fontsize=13, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(levels_labels)
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# Gate Count Comparison
ax2 = axes[0, 1]
iqm_gates = [results['IQM'][l]['gates'] for l in OPTIMIZATION_LEVELS]
ibm_gates = [results['IBM'][l]['gates'] for l in OPTIMIZATION_LEVELS]

ax2.bar(x - width/2, iqm_gates, width, label='IQM', color=colors_iqm, alpha=0.8, edgecolor='black')
ax2.bar(x + width/2, ibm_gates, width, label='IBM', color=colors_ibm, alpha=0.8, edgecolor='black')
ax2.set_ylabel('Gate Count', fontsize=12, fontweight='bold')
ax2.set_xlabel('Optimization Level', fontsize=12, fontweight='bold')
ax2.set_title('Gate Count Comparison', fontsize=13, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(levels_labels)
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

# Depth Reduction Percentage
ax3 = axes[1, 0]
iqm_depth_reduction = [(1 - results['IQM'][l]['depth'] / qc.depth()) * 100 for l in OPTIMIZATION_LEVELS]
ibm_depth_reduction = [(1 - results['IBM'][l]['depth'] / qc.depth()) * 100 for l in OPTIMIZATION_LEVELS]

ax3.plot(levels_labels, iqm_depth_reduction, marker='o', label='IQM', color=colors_iqm, linewidth=2, markersize=8)
ax3.plot(levels_labels, ibm_depth_reduction, marker='s', label='IBM', color=colors_ibm, linewidth=2, markersize=8)
ax3.axhline(y=0, color='black', linestyle='--', linewidth=1)
ax3.set_ylabel('Depth Reduction (%)', fontsize=12, fontweight='bold')
ax3.set_xlabel('Optimization Level', fontsize=12, fontweight='bold')
ax3.set_title('Depth Reduction Percentage', fontsize=13, fontweight='bold')
ax3.legend()
ax3.grid(alpha=0.3)

# Gate Reduction Percentage
ax4 = axes[1, 1]
iqm_gate_reduction = [(1 - results['IQM'][l]['gates'] / qc.size()) * 100 for l in OPTIMIZATION_LEVELS]
ibm_gate_reduction = [(1 - results['IBM'][l]['gates'] / qc.size()) * 100 for l in OPTIMIZATION_LEVELS]

ax4.plot(levels_labels, iqm_gate_reduction, marker='o', label='IQM', color=colors_iqm, linewidth=2, markersize=8)
ax4.plot(levels_labels, ibm_gate_reduction, marker='s', label='IBM', color=colors_ibm, linewidth=2, markersize=8)
ax4.axhline(y=0, color='black', linestyle='--', linewidth=1)
ax4.set_ylabel('Gate Reduction (%)', fontsize=12, fontweight='bold')
ax4.set_xlabel('Optimization Level', fontsize=12, fontweight='bold')
ax4.set_title('Gate Reduction Percentage', fontsize=13, fontweight='bold')
ax4.legend()
ax4.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Summary Statistics

In [ ]:
print("\n" + "=" * 80)
print("SUMMARY STATISTICS")
print("=" * 80)

print(f"\nOriginal Circuit: Depth={qc.depth()}, Gates={qc.size()}")

for provider in ['IQM', 'IBM']:
    print(f"\n{provider} Backend:")
    print("-" * 80)
    for level in OPTIMIZATION_LEVELS:
        depth = results[provider][level]['depth']
        gates = results[provider][level]['gates']
        depth_red = (1 - depth / qc.depth()) * 100
        gate_red = (1 - gates / qc.size()) * 100
        print(f"Level {level}: Depth={depth:3d} ({depth_red:+6.1f}%), Gates={gates:3d} ({gate_red:+6.1f}%)")

# Best optimization for each provider
print("\n" + "=" * 80)
print("BEST OPTIMIZATION (Level 3)")
print("=" * 80)

for provider in ['IQM', 'IBM']:
    depth = results[provider][3]['depth']
    gates = results[provider][3]['gates']
    depth_red = (1 - depth / qc.depth()) * 100
    gate_red = (1 - gates / qc.size()) * 100
    print(f"\n{provider}:")
    print(f"  Depth: {depth} ({depth_red:+.1f}% change)")
    print(f"  Gates: {gates} ({gate_red:+.1f}% change)")

## Measurement Results Comparison

In [ ]:
# Compare measurement distributions for Level 3
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Measurement Results - Optimization Level 3', fontsize=14, fontweight='bold')

for idx, provider in enumerate(['IQM', 'IBM']):
    ax = axes[idx]
    counts = results[provider][3]['counts']
    
    # Get top 10 results
    top_counts = dict(sorted(counts.items(), key=lambda x: x[1], reverse=True)[:10])
    states = list(top_counts.keys())
    values = [top_counts[s] / SHOTS * 100 for s in states]
    
    color = colors_iqm if provider == 'IQM' else colors_ibm
    ax.bar(range(len(states)), values, color=color, alpha=0.8, edgecolor='black')
    ax.set_ylabel('Probability (%)', fontsize=11, fontweight='bold')
    ax.set_xlabel('Measurement State', fontsize=11, fontweight='bold')
    ax.set_title(f'{provider} - Top 10 States', fontsize=12, fontweight='bold')
    ax.set_xticks(range(len(states)))
    ax.set_xticklabels(states, rotation=45, ha='right')
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()